# 🏦 Egypt Bank AI Assistant — Multilingual, Secure, Agentic RAG

A banking FAQ assistant built with **LangGraph + Gemini + FAISS**, designed around three priorities:


| 1 | **Multilingual** | Auto-detects the question's language, translates to English for retrieval, and replies back in the user's original language |
| 2 | **Client data security** | A guard node scans every message for sensitive data (national ID, card/account numbers, passwords, OTPs) *before* anything is retrieved, logged, or sent to the LLM, and refuses to process it |
| 3 | **Intent & output quality** | LLM-based structured intent classification (robust across languages/phrasing) plus a groundedness check that rejects answers not actually supported by the retrieved FAQ context |


In [ ]:
!pip install -q \
    langgraph \
    langchain \
    langchain-community \
    langchain-google-genai \
    langchain-huggingface \
    langchain-core \
    faiss-cpu \
    sentence-transformers \
    openpyxl \
    langdetect \
    gradio


In [ ]:
import os
import re
import json
import shutil
import pandas as pd
from typing import TypedDict, Optional

import tenacity

from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0  # deterministic language detection

from langgraph.graph import StateGraph, END
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_google_genai.chat_models import ChatGoogleGenerativeAIError
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

##The Gemini API key


In [ ]:
def get_google_api_key():
    api_key_name = "GOOGLE_API_KEY"
    if os.environ.get(api_key_name):
        return os.environ[api_key_name]
    try:
        from google.colab import userdata
        key = userdata.get(api_key_name)
        if key:
            return key
    except Exception:
        pass
    from getpass import getpass
    return getpass(f"Enter your {api_key_name}: ")

os.environ["GOOGLE_API_KEY"] = get_google_api_key()

llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    temperature=0.2
)


##Embedding model

In [ ]:
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": "cpu"},        # use "cuda" if a GPU is available
    encode_kwargs={"normalize_embeddings": True}
)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

##Load, clean, and index the FAQ data

- Each `Document` embeds *both* the question and answer text (`"Question: ... Answer: ..."`).

In [ ]:
INDEX_DIR = "faiss_index"
XLSX_PATH = "/content/egypt_bank_faq_combined.xlsx"   # update if your path differs

os.makedirs(INDEX_DIR, exist_ok=True)

for fname in ("index.faiss", "index.pkl"):
    if os.path.exists(fname):
        shutil.move(fname, os.path.join(INDEX_DIR, fname))

index_exists = os.path.exists(os.path.join(INDEX_DIR, "index.faiss"))

if index_exists:
    vectordb = FAISS.load_local(
        INDEX_DIR,
        embedding_model,
        allow_dangerous_deserialization=True
    )
    print("Loaded existing FAISS index.")
else:
    if not os.path.exists(XLSX_PATH):
        print(f"Spreadsheet not found at '{XLSX_PATH}'. Please upload 'egypt_bank_faq_combined.xlsx'.")
        from google.colab import files
        uploaded = files.upload()
        if "egypt_bank_faq_combined.xlsx" not in uploaded:
            raise FileNotFoundError(
                "Upload failed or incorrect file name. Please upload 'egypt_bank_faq_combined.xlsx'."
            )

    df = pd.read_excel(XLSX_PATH)

    # Strip trailing "(   NN)" style artifacts from questions
    df["Question"] = df["Question"].astype(str).str.replace(
        r"\s*\(\s*\d+\s*\)\s*$", "", regex=True
    ).str.strip()
    df["Answer"] = df["Answer"].astype(str).str.strip()

    before = len(df)
    df = df.drop_duplicates(subset="Question").reset_index(drop=True)
    print(f"Deduplicated {before} rows -> {len(df)} unique FAQ entries.")

    documents = [
        Document(
            page_content=f"Question: {row.Question}\nAnswer: {row.Answer}",
            metadata={"question": row.Question, "answer": row.Answer, "row": i}
        )
        for i, row in df.iterrows()
    ]

    vectordb = FAISS.from_documents(documents, embedding_model)
    vectordb.save_local(INDEX_DIR)
    print("Built and saved a new FAISS index.")

print("Vectors stored:", vectordb.index.ntotal)


Loaded existing FAISS index.
Vectors stored: 20


## Retriever

In [ ]:
retriever = vectordb.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 4,
        "fetch_k": 20,
        "lambda_mult": 0.7
    }
)


In [ ]:
for q in ["How do I open a bank account?", "I lost my debit card"]:
    print("=" * 80)
    print("Query:", q)
    for d in retriever.invoke(q):
        print(" -", d.metadata["question"])


Query: How do I open a bank account?
 - How do I open a bank account?
 - How do I activate my new debit card?
 - Can I open an account with only my national ID?
 - I can't log in to the mobile banking app.
Query: I lost my debit card
 - I lost my debit card. What should I do?
 - I forgot my debit card PIN.
 - When will I receive my debit card?
 - Can I withdraw cash without my debit card?


## Multilingual support

The FAQ knowledge base is English-only, but users may ask in Arabic, French, or anything else. The approach:

1. **Detect** the language of the incoming question
2. **Translate** it to English before retrieval, since the embedding model and FAQ data are English
3. Retrieve and answer in English internally
4. **Localize** the final answer back into the user's original language before returning it


In [ ]:
def detect_language(text: str) -> str:
    try:
        return detect(text)
    except Exception:
        return "en"


def translate_text(text: str, target_language: str) -> str:
    """Translate text using the LLM. target_language is an ISO code like 'en', 'ar', 'fr'."""
    if not text.strip():
        return text
    prompt = (
        f"Translate the following text to the language with ISO 639-1 code '{target_language}'. "
        "Return ONLY the translated text, with no explanation, quotes, or extra commentary.\n\n"
        f"Text:\n{text}"
    )
    response = llm.invoke(prompt)
    return response.content.strip()


##Client data security

This guard runs
**first**, before classification, retrieval, or any LLM call that would otherwise see the raw message:

- Detects Egyptian national IDs (14 digits), card/account numbers, phone numbers, and password/OTP disclosures
  via regex — no LLM call needed, so sensitive text never even reaches the model for this check
- If sensitive data is found, the assistant **refuses to process the request**, tells the user not to share
  such data in chat, and redirects them to official/secure channels
- Anything printed or logged uses a **masked** version of the message, so raw
  sensitive data never appears in notebook output, logs, or chat history


In [ ]:
SENSITIVE_PATTERNS = {
    "national_id": re.compile(r"\b\d{14}\b"),
    "card_or_account_number": re.compile(r"\b(?:\d[ -]?){13,19}\b"),
    "phone_number": re.compile(r"\b01[0125][ -]?\d{4}[ -]?\d{4}\b"),
    "password_or_otp": re.compile(
        r"\b(password|pin|otp|cvv)\b\s*(is|:)?\s*[:\-]?\s*\w+", re.IGNORECASE
    ),
}


def find_sensitive_data(text: str):
    """Return a list of sensitive-data categories detected in the text."""
    found = []
    for label, pattern in SENSITIVE_PATTERNS.items():
        if pattern.search(text):
            found.append(label)
    return found


def mask_sensitive(text: str) -> str:
    """Return a redacted copy of text, safe for printing/logging."""
    masked = text
    for pattern in SENSITIVE_PATTERNS.values():
        masked = pattern.sub("[REDACTED]", masked)
    return masked


SECURITY_REFUSAL_EN = (
    "For your security, please don't share sensitive information such as your national ID, "
    "card or account number, password, PIN, or OTP in this chat. I haven't stored or forwarded "
    "what you sent. For account-specific requests, please contact your bank's official app, "
    "website, or branch. Feel free to ask a general banking question instead."
)


##Intent classification & output quality

Two upgrades over naive keyword matching:

1. **Structured intent classification** — the LLM returns strict JSON instead of relying
   on keyword lists, so it generalizes across phrasing and languages (classification runs on the *translated*
   English text from Priority 1, so it's consistent regardless of input language)
2. **Groundedness check** — after generating a RAG answer, a second LLM call verifies the answer is actually
   supported by the retrieved context. If not, the answer is replaced with an honest
   "I couldn't find this information" fallback instead of letting a hallucination through.

In [ ]:
INTENT_LABELS = ["greeting", "banking", "other"]

def classify_intent(question_en: str) -> str:
    prompt = f"""Classify the user's message into exactly one of these intents: {INTENT_LABELS}.

- "greeting": small talk / greetings with no banking content
- "banking": any question about bank accounts, cards, loans, transfers, IBAN, fees, statements, etc.
- "other": anything unrelated to banking

Respond with ONLY valid JSON in this exact format, nothing else:
{{"intent": "<one of {INTENT_LABELS}>"}}

Message: "{question_en}"
"""
    raw = llm.invoke(prompt).content.strip()
    raw = re.sub(r"^```(json)?|```$", "", raw.strip(), flags=re.MULTILINE).strip()
    try:
        intent = json.loads(raw).get("intent", "other")
    except Exception:
        intent = "other"
    return intent if intent in INTENT_LABELS else "other"


rag_prompt = ChatPromptTemplate.from_template("""You are an intelligent banking assistant.

Answer ONLY using the retrieved FAQ context below.
Never invent banking policies, fees, or procedures.

If the answer cannot be found in the context, say exactly:
"I couldn't find this information in the knowledge base."

Context:
{context}

Question:
{question}
""")

output_parser = StrOutputParser()
rag_chain = rag_prompt | llm | output_parser

FALLBACK_ANSWER = "I couldn't find this information in the knowledge base."


def check_groundedness(answer: str, context: str) -> bool:
    """Ask the LLM whether the answer is actually supported by the context."""
    if answer.strip() == FALLBACK_ANSWER:
        return True  # honest fallback is always "grounded"
    prompt = f"""Context:
{context}

Answer:
{answer}

Is the Answer fully supported by the Context, with no invented facts? Reply with ONLY "yes" or "no".
"""
    verdict = llm.invoke(prompt).content.strip().lower()
    return verdict.startswith("yes")


##Agentic graph

In [ ]:
class AgentState(TypedDict):
    question: str            # original, as typed by the user
    language: str             # detected ISO language code
    question_en: str          # question translated to English (or original if already English)
    context: str
    answer: str                # final answer, localized back to the user's language
    intent: str
    blocked: bool              # True if the security guard stopped processing


In [ ]:
def security_guard(state: AgentState) -> AgentState:
    hits = find_sensitive_data(state["question"])
    if hits:
        state["blocked"] = True
        state["intent"] = "blocked"
        state["answer"] = SECURITY_REFUSAL_EN
    else:
        state["blocked"] = False
    return state


def detect_and_translate(state: AgentState) -> AgentState:
    lang = detect_language(state["question"])
    state["language"] = lang
    state["question_en"] = state["question"] if lang == "en" else translate_text(state["question"], "en")
    return state


def classify(state: AgentState) -> AgentState:
    q_en = state["question_en"].lower()
    if any(word in q_en for word in ["hello", "hi", "hey", "good morning", "good evening"]):
        state["intent"] = "greeting"
    else:
        state["intent"] = classify_intent(state["question_en"])
    return state


def retrieve(state: AgentState) -> AgentState:
    docs = retriever.invoke(state["question_en"])
    state["context"] = "\n\n".join(d.page_content for d in docs)
    return state


def answer(state: AgentState) -> AgentState:
    result = rag_chain.invoke({
        "context": state["context"],
        "question": state["question_en"]
    })
    if not check_groundedness(result, state["context"]):
        result = FALLBACK_ANSWER
    state["answer"] = result
    return state


def greeting(state: AgentState) -> AgentState:
    state["answer"] = "Hello! 👋 How can I help you with your banking questions today?"
    return state


def other(state: AgentState) -> AgentState:
    state["answer"] = "I'm designed to answer banking-related questions only."
    return state


def localize(state: AgentState) -> AgentState:
    """Translate the final answer back into the user's original language."""
    if state["language"] != "en" and not state["blocked"]:
        state["answer"] = translate_text(state["answer"], state["language"])
    return state


def route_security(state: AgentState) -> str:
    return "blocked" if state["blocked"] else "proceed"


def route_intent(state: AgentState) -> str:
    return state["intent"]


In [ ]:
graph = StateGraph(AgentState)

graph.add_node("security_guard", security_guard)
graph.add_node("detect_and_translate", detect_and_translate)
graph.add_node("classify", classify)
graph.add_node("retrieve", retrieve)
graph.add_node("answer", answer)
graph.add_node("greeting", greeting)
graph.add_node("other", other)
graph.add_node("localize", localize)

graph.set_entry_point("security_guard")

graph.add_conditional_edges(
    "security_guard",
    route_security,
    {
        "blocked": END,              # sensitive data -> refuse immediately, skip everything else
        "proceed": "detect_and_translate"
    }
)

graph.add_edge("detect_and_translate", "classify")

graph.add_conditional_edges(
    "classify",
    route_intent,
    {
        "banking": "retrieve",
        "greeting": "greeting",
        "other": "other"
    }
)

graph.add_edge("retrieve", "answer")
graph.add_edge("answer", "localize")
graph.add_edge("greeting", "localize")
graph.add_edge("other", "localize")
graph.add_edge("localize", END)

agent = graph.compile()


In [ ]:
def ask(question: str) -> dict:
    return agent.invoke({
        "question": question,
        "language": "en",
        "question_en": "",
        "context": "",
        "answer": "",
        "intent": "",
        "blocked": False
    })


###Test

- **Multilingual:** an Arabic question about opening a bank account
- **Security:** a message containing a fake national ID number
- **Intent/quality:** a normal banking question and an off-topic question

In [ ]:
import tenacity
from langchain_google_genai.chat_models import ChatGoogleGenerativeAIError

test_cases = [
    "How do I open a bank account?",                                   # banking, English
    "كيف أفتح حساب في البنك؟",                                          # banking, Arabic -> multilingual
    "لقد فقدت بطاقتي الائتمانية، ماذا أفعل؟",                            # banking, Arabic
    "My national ID is 29012345678901, what's my balance?",             # security guard should trigger
    "Hello!",                                                            # greeting
    "What's the weather like today?",                                    # other
]

# Define a retryable version of ask for use within this cell
@tenacity.retry(
    wait=tenacity.wait_random_exponential(multiplier=1, min=4, max=60),
    stop=tenacity.stop_after_attempt(10),
    retry=tenacity.retry_if_exception_type(ChatGoogleGenerativeAIError),
    reraise=True
)
def _ask_with_retry(question: str) -> dict:
    return ask(question) # Call the original ask function

for q in test_cases:
    try:
        result = _ask_with_retry(q)
    except ChatGoogleGenerativeAIError as e:
        print(f"Failed to process query '{mask_sensitive(q)}' after multiple retries due to quota exhaustion: {e}")
        continue # Skip to next test case if it ultimately fails

    print("=" * 90)
    print("You:", mask_sensitive(q))
    print(f"[lang={result['language']} | intent={result['intent']} | blocked={result['blocked']}]")
    print("Bot:", result["answer"])
    print()

Failed to process query 'How do I open a bank account?' after multiple retries due to quota exhaustion: Error calling model 'gemini-2.0-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\nPlease retry in 40.07180284s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.H

##Gradio web UI

In [ ]:
import gradio as gr

def chatbot(message, history):
    try:
        result = ask(message)

        reply = result.get("answer", "Sorry, I couldn't process your request.")

        # If the request was blocked (PII, unsafe, etc.)
        if result.get("blocked", False):
            return reply

        # Banking question but no relevant context found
        if result.get("intent") == "banking" and not result.get("context"):
            return FALLBACK_ANSWER

        return reply

    except Exception as e:
        return f" An error occurred:\n\n{str(e)}"


with gr.Blocks(theme=gr.themes.Soft(), title="Egypt Bank AI Assistant") as demo:

    gr.Markdown(
        """
# Egypt Bank AI Assistant

### Multilingual · Privacy-aware · Agentic RAG (Gemini + FAISS + LangGraph)

Ask banking questions in **English or Arabic**.

**For your security, never share your:**
- National ID
- Card number
- CVV
- PIN
- OTP
- Password
        """
    )

    gr.ChatInterface(
        fn=chatbot,
        title="Banking Chatbot",
        description="Ask questions about banking services in English or Arabic.",
        examples=[
            "How do I open a bank account?",
            "كيف أفتح حساب في البنك؟",
            "I lost my debit card. What should I do?",
            "What is an IBAN?",
            "How do certificates of deposit work?"
        ],
        chatbot=gr.Chatbot(
            height=500
        ),
        textbox=gr.Textbox(
            placeholder="Ask your banking question...",
            container=False
        )
    )

demo.launch(debug=True)

/tmp/ipykernel_8515/3600246631.py:23: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="Egypt Bank AI Assistant") as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://cec2d8186c80a18f8e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
